In [ ]:
# 07 — ADAPTIVE / GATED FUSION
# Spatial + Frequency Adaptive Fusion

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import copy
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from PIL import Image
import pywt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

PROJECT_DIR = "/content/drive/MyDrive/Multimedia_Authentication"

FACES_DIR = os.path.join(
    PROJECT_DIR,
    "faces"
)

SPLIT_FILE = os.path.join(
    PROJECT_DIR,
    "splits",
    "ffpp_2647_processed_split.csv"
)

SPATIAL_CHECKPOINT = os.path.join(
    PROJECT_DIR,
    "spatial_resnet18_best.pth"
)

FREQUENCY_CHECKPOINT = os.path.join(
    PROJECT_DIR,
    "frequency_resnet18_best.pth"
)

FIXED_FUSION_CHECKPOINT = os.path.join(
    PROJECT_DIR,
    "models",
    "fusion_resnet18_auc_best.pth"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

print("Project exists:", os.path.exists(PROJECT_DIR))
print("Faces exists:", os.path.exists(FACES_DIR))
print("Split exists:", os.path.exists(SPLIT_FILE))

print(
    "Spatial checkpoint:",
    os.path.exists(SPATIAL_CHECKPOINT)
)

print(
    "Frequency checkpoint:",
    os.path.exists(FREQUENCY_CHECKPOINT)
)

print(
    "Fixed fusion checkpoint:",
    os.path.exists(FIXED_FUSION_CHECKPOINT)
)

Device: cuda
Project exists: True
Faces exists: True
Split exists: True
Spatial checkpoint: True
Frequency checkpoint: True
Fixed fusion checkpoint: True


In [ ]:
df = pd.read_csv(SPLIT_FILE)

print("Total rows:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nSplit distribution:")
print(df["split"].value_counts())

print("\nLabel distribution:")
print(df["Label"].value_counts())

df["face_file"] = df["npz_name"]

# Build split dataframes

train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

# Verify face files exist

def filter_existing_faces(dataframe):
    mask = dataframe["face_file"].apply(
        lambda x: os.path.exists(
            os.path.join(FACES_DIR, x)
        )
    )
    return dataframe[mask].reset_index(drop=True)

train_df = filter_existing_faces(train_df)
val_df   = filter_existing_faces(val_df)
test_df  = filter_existing_faces(test_df)

print("\nAfter face-file verification:")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

print("\nLabels after verification:")

print("\nTrain:")
print(train_df["Label"].value_counts())

print("\nVal:")
print(val_df["Label"].value_counts())

print("\nTest:")
print(test_df["Label"].value_counts())

Total rows: 2647

Columns:
['Unnamed: 0', 'File Path', 'Label', 'Frame Count', 'Width', 'Height', 'Codec', 'File Size(MB)', 'source_id', 'split', 'npz_name', 'extracted']

Split distribution:
split
train    1822
test      424
val       401
Name: count, dtype: int64

Label distribution:
Label
FAKE    2211
REAL     436
Name: count, dtype: int64

After face-file verification:
Train: 1770
Val  : 392
Test : 418

Labels after verification:

Train:
Label
FAKE    1472
REAL     298
Name: count, dtype: int64

Val:
Label
FAKE    324
REAL     68
Name: count, dtype: int64

Test:
Label
FAKE    354
REAL     64
Name: count, dtype: int64


In [ ]:

# 3. FREQUENCY PREPROCESSING

def dwt_high_frequency(frame):
    """
    Extract high-frequency information using Haar DWT.

    For each RGB channel:
        - Compute 2D Haar DWT
        - Keep LH, HL, HH high-frequency subbands

    Then average each subband across RGB channels.
    """

    frame = frame.astype(np.float32) / 255.0

    LH_channels = []
    HL_channels = []
    HH_channels = []

    for c in range(3):
        channel = frame[:, :, c]

        LL, (LH, HL, HH) = pywt.dwt2(
            channel,
            "haar"
        )

        LH_channels.append(LH)
        HL_channels.append(HL)
        HH_channels.append(HH)

    LH = np.mean(
        np.stack(LH_channels),
        axis=0
    )

    HL = np.mean(
        np.stack(HL_channels),
        axis=0
    )

    HH = np.mean(
        np.stack(HH_channels),
        axis=0
    )

    return np.stack(
        [LH, HL, HH],
        axis=-1
    ).astype(np.float32)


def normalize_frequency_image(freq_img):
    """
    Min-max normalize DWT high-frequency image
    and convert it into a PIL image.
    """

    min_val = freq_img.min()
    max_val = freq_img.max()

    freq_img = (
        (freq_img - min_val)
        / (max_val - min_val + 1e-8)
    )

    freq_img = (
        freq_img * 255
    ).clip(0, 255).astype(np.uint8)

    return Image.fromarray(freq_img)


print("Frequency preprocessing functions loaded successfully.")

Frequency preprocessing functions loaded successfully.


In [ ]:

# 4. SPATIAL + FREQUENCY DATASET

class SpatialFrequencyDataset(Dataset):

    def __init__(
        self,
        dataframe,
        faces_dir,
        rgb_transform=None,
        frequency_transform=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.faces_dir = faces_dir
        self.rgb_transform = rgb_transform
        self.frequency_transform = frequency_transform

        self.label_map = {
            "REAL": 0,
            "FAKE": 1
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        npz_path = os.path.join(
            self.faces_dir,
            row["face_file"]
        )

        data = np.load(npz_path)
        frames = data["frames"]

        rgb_frames = []
        frequency_frames = []

        for frame in frames:

            # RGB / spatial branch

            rgb_image = Image.fromarray(frame)

            if self.rgb_transform:
                rgb_image = self.rgb_transform(
                    rgb_image
                )

            rgb_frames.append(rgb_image)

            # Frequency branch

            freq = dwt_high_frequency(frame)

            freq_image = normalize_frequency_image(
                freq
            )

            freq_image = freq_image.resize(
                (224, 224)
            )

            if self.frequency_transform:
                freq_image = self.frequency_transform(
                    freq_image
                )

            frequency_frames.append(
                freq_image
            )

        rgb_frames = torch.stack(
            rgb_frames
        )

        frequency_frames = torch.stack(
            frequency_frames
        )

        label = self.label_map[
            row["Label"]
        ]

        return (
            rgb_frames,
            frequency_frames,
            torch.tensor(
                label,
                dtype=torch.long
            )
        )


print("SpatialFrequencyDataset defined.")

SpatialFrequencyDataset defined.


In [ ]:

# 5. IMAGE TRANSFORM

rgb_transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

frequency_transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("RGB transform ready.")
print("Frequency transform ready.")

RGB transform ready.
Frequency transform ready.


In [ ]:
# ============================================================
# 6. CREATE DATASETS
# ============================================================

train_dataset = SpatialFrequencyDataset(
    train_df,
    FACES_DIR,
    rgb_transform=rgb_transform,
    frequency_transform=frequency_transform
)

val_dataset = SpatialFrequencyDataset(
    val_df,
    FACES_DIR,
    rgb_transform=rgb_transform,
    frequency_transform=frequency_transform
)

test_dataset = SpatialFrequencyDataset(
    test_df,
    FACES_DIR,
    rgb_transform=rgb_transform,
    frequency_transform=frequency_transform
)

print("Train dataset:", len(train_dataset))
print("Val dataset  :", len(val_dataset))
print("Test dataset :", len(test_dataset))

Train dataset: 1770
Val dataset  : 392
Test dataset : 418


In [ ]:

# 7. DATALOADERS

BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Batch size:", BATCH_SIZE)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

Batch size: 4
Train batches: 443
Val batches  : 98
Test batches : 105


In [ ]:

# 9. RESNET18 FEATURE EXTRACTON

class ResNet18FeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()

        backbone = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        # Remove final classification layer
        self.features = nn.Sequential(
            *list(backbone.children())[:-1]
        )

    def forward(self, x):

        x = self.features(x)

        x = torch.flatten(
            x,
            start_dim=1
        )

        return x


print("ResNet18 feature extractor defined.")
print("Output feature dimension: 512")

ResNet18 feature extractor defined.
Output feature dimension: 512


In [ ]:

# 10. ADAPTIVE / GATED FUSION MODEL

class GatedSpatialFrequencyFusion(nn.Module):

    def __init__(self):
        super().__init__()

        # Two pretrained branches

        self.spatial_branch = ResNet18FeatureExtractor()

        self.frequency_branch = ResNet18FeatureExtractor()

        # Adaptive gate
        #
        # Input:
        #   spatial (512) + frequency (512) = 1024
        #
        # Output:
        #   one scalar alpha in [0,1]

        self.gate = nn.Sequential(

            nn.Linear(1024, 128),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(128, 1),

            nn.Sigmoid()
        )

        # Start with equal contribution from both branches.
        # Zero weights/bias in the final gate layer gives:
        # sigmoid(0) = 0.5

        nn.init.zeros_(self.gate[-2].weight)
        nn.init.zeros_(self.gate[-2].bias)

        # Classifier

        self.classifier = nn.Sequential(

            nn.Linear(512, 256),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(256, 2)
        )

    def extract_video_features(
        self,
        frames,
        branch
    ):

        B, T, C, H, W = frames.shape

        # Process all frames through ResNet
        frames = frames.view(
            B * T,
            C,
            H,
            W
        )

        features = branch(frames)

        # Restore temporal dimension
        features = features.view(
            B,
            T,
            512
        )

        # Temporal mean pooling
        features = features.mean(
            dim=1
        )

        return features

    def forward(
        self,
        rgb_frames,
        frequency_frames,
        return_gate=False
    ):

        # Extract spatial features

        spatial_features = self.extract_video_features(
            rgb_frames,
            self.spatial_branch
        )

        # Extract frequency features

        frequency_features = self.extract_video_features(
            frequency_frames,
            self.frequency_branch
        )

        # Compute adaptive gate

        combined_features = torch.cat(
            [
                spatial_features,
                frequency_features
            ],
            dim=1
        )

        alpha = self.gate(
            combined_features
        )

        # Adaptive fusion

        fused_features = (
            alpha * spatial_features
            +
            (1.0 - alpha) * frequency_features
        )

        # Classification

        logits = self.classifier(
            fused_features
        )

        if return_gate:
            return logits, alpha

        return logits


print("GatedSpatialFrequencyFusion defined.")

GatedSpatialFrequencyFusion defined.


In [ ]:

# 11. INITIALIZE MODEL

model = GatedSpatialFrequencyFusion()

model = model.to(device)

print(model)

print("\nModel device:", next(model.parameters()).device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s]


GatedSpatialFrequencyFusion(
  (spatial_branch): ResNet18FeatureExtractor(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=Fals

In [ ]:

# 12. LOAD PRETRAINED SPATIAL + FREQUENCY BRANCH

def load_branch_checkpoint(branch, checkpoint_path, branch_name):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    # Handle checkpoints saved either as:
    # 1. plain state_dict
    # 2. {"model_state_dict": ...}
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    # Remove possible prefixes if the checkpoint came
    # from a larger model.
    cleaned_state_dict = {}

    for key, value in state_dict.items():

        if key.startswith("features."):
            new_key = key

        elif key.startswith("backbone."):
            new_key = key.replace(
                "backbone.",
                "features.",
                1
            )

        else:
            new_key = key

        cleaned_state_dict[new_key] = value

    result = branch.load_state_dict(
        cleaned_state_dict,
        strict=False
    )

    print(f"\n{branch_name} checkpoint loaded:")
    print("  Path:", checkpoint_path)
    print("  Missing keys:", len(result.missing_keys))
    print("  Unexpected keys:", len(result.unexpected_keys))

    if result.missing_keys:
        print("  Missing:", result.missing_keys[:10])

    if result.unexpected_keys:
        print("  Unexpected:", result.unexpected_keys[:10])


load_branch_checkpoint(
    model.spatial_branch,
    SPATIAL_CHECKPOINT,
    "Spatial"
)

load_branch_checkpoint(
    model.frequency_branch,
    FREQUENCY_CHECKPOINT,
    "Frequency"
)


Spatial checkpoint loaded:
  Path: /content/drive/MyDrive/Multimedia_Authentication/spatial_resnet18_best.pth
  Missing keys: 0
  Unexpected keys: 2
  Unexpected: ['classifier.weight', 'classifier.bias']

Frequency checkpoint loaded:
  Path: /content/drive/MyDrive/Multimedia_Authentication/frequency_resnet18_best.pth
  Missing keys: 0
  Unexpected keys: 2
  Unexpected: ['classifier.weight', 'classifier.bias']


In [ ]:

# 13. FREEZE PRETRAINED BRANCHES

for param in model.spatial_branch.parameters():
    param.requires_grad = False

for param in model.frequency_branch.parameters():
    param.requires_grad = False

# Gate and classifier remain trainable
for param in model.gate.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True


# Verify trainable parameters

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = sum(
    p.numel()
    for p in model.parameters()
    if not p.requires_grad
)

print("Trainable parameters:", trainable_params)
print("Frozen parameters   :", frozen_params)

print("\nSpatial branch trainable:",
      any(p.requires_grad for p in model.spatial_branch.parameters()))

print("Frequency branch trainable:",
      any(p.requires_grad for p in model.frequency_branch.parameters()))

print("Gate trainable:",
      any(p.requires_grad for p in model.gate.parameters()))

print("Classifier trainable:",
      any(p.requires_grad for p in model.classifier.parameters()))

Trainable parameters: 263171
Frozen parameters   : 22353024

Spatial branch trainable: False
Frequency branch trainable: False
Gate trainable: True
Classifier trainable: True


In [ ]:

# 14. LOSS FUNCTION WITH CLASS WEIGHTS

real_count = (train_df["Label"] == "REAL").sum()
fake_count = (train_df["Label"] == "FAKE").sum()

total_count = real_count + fake_count

real_weight = total_count / (2 * real_count)
fake_weight = total_count / (2 * fake_count)

class_weights = torch.tensor(
    [real_weight, fake_weight],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

print("REAL count:", real_count)
print("FAKE count:", fake_count)

print("\nClass weights:")
print(class_weights)

REAL count: 298
FAKE count: 1472

Class weights:
tensor([2.9698, 0.6012], device='cuda:0')


In [ ]:

# 15. GATED FUSION WARMUP OPTIMIZER

WARMUP_LR = 1e-3

warmup_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=WARMUP_LR,
    weight_decay=1e-4
)

print("Warmup learning rate:", WARMUP_LR)

print(
    "Trainable tensors:",
    sum(
        1 for p in model.parameters()
        if p.requires_grad
    )
)

Warmup learning rate: 0.001
Trainable tensors: 8


In [ ]:

# 16. FORWARD-PASS SANITY CHECK

model.eval()

# Get one fresh batch
rgb_batch, freq_batch, label_batch = next(
    iter(train_loader)
)

with torch.no_grad():

    rgb_batch = rgb_batch.to(device)
    freq_batch = freq_batch.to(device)
    label_batch = label_batch.to(device)

    logits, alpha = model(
        rgb_batch,
        freq_batch,
        return_gate=True
    )

print("RGB input shape      :", rgb_batch.shape)
print("Frequency input shape:", freq_batch.shape)
print("Label shape           :", label_batch.shape)

print("\nLogits shape:", logits.shape)
print("Gate shape  :", alpha.shape)

print("\nGate values:")
print(alpha.squeeze(1).cpu().numpy())

print("\nLogits:")
print(logits.cpu().numpy())

print("\nGate min:", alpha.min().item())
print("Gate max:", alpha.max().item())

RGB input shape      : torch.Size([4, 8, 3, 224, 224])
Frequency input shape: torch.Size([4, 8, 3, 224, 224])
Label shape           : torch.Size([4])

Logits shape: torch.Size([4, 2])
Gate shape  : torch.Size([4, 1])

Gate values:
[0.5 0.5 0.5 0.5]

Logits:
[[-0.04191468  0.01675044]
 [-0.3797429   0.3933356 ]
 [ 0.00737483  0.06841793]
 [ 0.06027819  0.13114467]]

Gate min: 0.5
Gate max: 0.5


In [ ]:

# 17. TRAINING AND EVALUATION FUNCTIONS

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    for rgb_frames, freq_frames, labels in loader:

        rgb_frames = rgb_frames.to(device)
        freq_frames = freq_frames.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(
            rgb_frames,
            freq_frames
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        probs = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        preds = torch.argmax(
            logits,
            dim=1
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_probs.extend(
            probs.detach().cpu().numpy()
        )

    return {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            all_labels,
            all_preds
        ),
        "precision": precision_score(
            all_labels,
            all_preds,
            zero_division=0
        ),
        "recall": recall_score(
            all_labels,
            all_preds,
            zero_division=0
        ),
        "f1": f1_score(
            all_labels,
            all_preds,
            zero_division=0
        ),
        "auc": roc_auc_score(
            all_labels,
            all_probs
        )
    }


def evaluate_fusion(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []
    all_gates = []

    with torch.no_grad():

        for rgb_frames, freq_frames, labels in loader:

            rgb_frames = rgb_frames.to(device)
            freq_frames = freq_frames.to(device)
            labels = labels.to(device)

            logits, alpha = model(
                rgb_frames,
                freq_frames,
                return_gate=True
            )

            loss = criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item() * labels.size(0)
            )

            probs = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_gates.extend(
                alpha.squeeze(1).cpu().numpy()
            )

    return {
        "loss": running_loss / len(loader.dataset),

        "accuracy": accuracy_score(
            all_labels,
            all_preds
        ),

        "precision": precision_score(
            all_labels,
            all_preds,
            zero_division=0
        ),

        "recall": recall_score(
            all_labels,
            all_preds,
            zero_division=0
        ),

        "f1": f1_score(
            all_labels,
            all_preds,
            zero_division=0
        ),

        "auc": roc_auc_score(
            all_labels,
            all_probs
        ),

        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs),
        "gates": np.array(all_gates)
    }


print("Training and evaluation functions ready.")

Training and evaluation functions ready.


In [ ]:
# ============================================================
# 17. FEATURE EXTRACTION MODEL
# ============================================================

class ResNet18FeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()

        backbone = resnet18(
            weights=None
        )

        self.features = nn.Sequential(
            *list(backbone.children())[:-1]
        )

    def forward(self, x):

        x = self.features(x)

        x = torch.flatten(
            x,
            start_dim=1
        )

        return x


print("Feature extractor ready.")

Feature extractor ready.


In [ ]:

# 18. LOAD TRAINED BRANCHES

spatial_extractor = ResNet18FeatureExtractor().to(device)
frequency_extractor = ResNet18FeatureExtractor().to(device)


def load_extractor(extractor, checkpoint_path, name):

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    cleaned = {}

    for key, value in state_dict.items():

        if key.startswith("features."):
            cleaned[key] = value

        elif key.startswith("backbone."):
            cleaned[
                key.replace(
                    "backbone.",
                    "features.",
                    1
                )
            ] = value

    result = extractor.load_state_dict(
        cleaned,
        strict=False
    )

    print(
        f"{name}: "
        f"missing={len(result.missing_keys)}, "
        f"unexpected={len(result.unexpected_keys)}"
    )


load_extractor(
    spatial_extractor,
    SPATIAL_CHECKPOINT,
    "Spatial"
)

load_extractor(
    frequency_extractor,
    FREQUENCY_CHECKPOINT,
    "Frequency"
)

spatial_extractor.eval()
frequency_extractor.eval()

for param in spatial_extractor.parameters():
    param.requires_grad = False

for param in frequency_extractor.parameters():
    param.requires_grad = False

print("Branches frozen.")

Spatial: missing=0, unexpected=0
Frequency: missing=0, unexpected=0
Branches frozen.


In [ ]:

# 19. PRECOMPUTE VIDEO FEATURES

def extract_features(loader, split_name):

    spatial_features = []
    frequency_features = []
    labels_list = []

    start_time = time.time()

    with torch.no_grad():

        for batch_idx, (
            rgb_frames,
            freq_frames,
            labels
        ) in enumerate(loader):

            rgb_frames = rgb_frames.to(device)
            freq_frames = freq_frames.to(device)

            B, T, C, H, W = rgb_frames.shape

            # Spatial branch

            rgb_flat = rgb_frames.view(
                B * T,
                C,
                H,
                W
            )

            spatial = spatial_extractor(
                rgb_flat
            )

            spatial = spatial.view(
                B,
                T,
                512
            )

            spatial = spatial.mean(
                dim=1
            )

            # Frequency branch

            freq_flat = freq_frames.view(
                B * T,
                C,
                H,
                W
            )

            frequency = frequency_extractor(
                freq_flat
            )

            frequency = frequency.view(
                B,
                T,
                512
            )

            frequency = frequency.mean(
                dim=1
            )

            spatial_features.append(
                spatial.cpu()
            )

            frequency_features.append(
                frequency.cpu()
            )

            labels_list.append(
                labels.cpu()
            )

            if (
                batch_idx + 1
            ) % 50 == 0:

                print(
                    f"{split_name}: "
                    f"{batch_idx+1}/{len(loader)} batches"
                )

    spatial_features = torch.cat(
        spatial_features
    )

    frequency_features = torch.cat(
        frequency_features
    )

    labels_tensor = torch.cat(
        labels_list
    )

    elapsed = time.time() - start_time

    print(
        f"\n{split_name} extraction complete."
    )

    print(
        "Spatial shape   :",
        spatial_features.shape
    )

    print(
        "Frequency shape :",
        frequency_features.shape
    )

    print(
        "Labels shape    :",
        labels_tensor.shape
    )

    print(
        "Time:",
        round(elapsed / 60, 2),
        "minutes"
    )

    return (
        spatial_features,
        frequency_features,
        labels_tensor
    )

In [ ]:

# 20. EXTRACT FEATURES FOR ALL SPLITS

train_spatial, train_frequency, train_labels = (
    extract_features(
        train_loader,
        "Train"
    )
)

val_spatial, val_frequency, val_labels = (
    extract_features(
        val_loader,
        "Validation"
    )
)

test_spatial, test_frequency, test_labels = (
    extract_features(
        test_loader,
        "Test"
    )
)

Train: 50/443 batches
Train: 100/443 batches
Train: 150/443 batches
Train: 200/443 batches
Train: 250/443 batches
Train: 300/443 batches
Train: 350/443 batches
Train: 400/443 batches

Train extraction complete.
Spatial shape   : torch.Size([1770, 512])
Frequency shape : torch.Size([1770, 512])
Labels shape    : torch.Size([1770])
Time: 77.2 minutes
Validation: 50/98 batches

Validation extraction complete.
Spatial shape   : torch.Size([392, 512])
Frequency shape : torch.Size([392, 512])
Labels shape    : torch.Size([392])
Time: 17.02 minutes
Test: 50/105 batches
Test: 100/105 batches

Test extraction complete.
Spatial shape   : torch.Size([418, 512])
Frequency shape : torch.Size([418, 512])
Labels shape    : torch.Size([418])
Time: 17.77 minutes


In [ ]:

# 21. SAVE PRECOMPUTED FEATURES

FEATURE_DIR = os.path.join(
    PROJECT_DIR,
    "features"
)

os.makedirs(
    FEATURE_DIR,
    exist_ok=True
)

torch.save(
    {
        "spatial": train_spatial,
        "frequency": train_frequency,
        "labels": train_labels
    },
    os.path.join(
        FEATURE_DIR,
        "gated_train_features.pt"
    )
)

torch.save(
    {
        "spatial": val_spatial,
        "frequency": val_frequency,
        "labels": val_labels
    },
    os.path.join(
        FEATURE_DIR,
        "gated_val_features.pt"
    )
)

torch.save(
    {
        "spatial": test_spatial,
        "frequency": test_frequency,
        "labels": test_labels
    },
    os.path.join(
        FEATURE_DIR,
        "gated_test_features.pt"
    )
)

print("Features saved successfully.")
print("Directory:", FEATURE_DIR)

Features saved successfully.
Directory: /content/drive/MyDrive/Multimedia_Authentication/features


In [24]:

FEATURE_DIR = os.path.join(PROJECT_DIR, "features")

train_features = torch.load(
    os.path.join(FEATURE_DIR, "gated_train_features.pt"),
    map_location="cpu"
)
val_features = torch.load(
    os.path.join(FEATURE_DIR, "gated_val_features.pt"),
    map_location="cpu"
)
test_features = torch.load(
    os.path.join(FEATURE_DIR, "gated_test_features.pt"),
    map_location="cpu"
)

train_spatial = train_features["spatial"]
train_frequency = train_features["frequency"]
train_labels = train_features["labels"]

val_spatial = val_features["spatial"]
val_frequency = val_features["frequency"]
val_labels = val_features["labels"]

test_spatial = test_features["spatial"]
test_frequency = test_features["frequency"]
test_labels = test_features["labels"]

print("Train:", train_spatial.shape, train_frequency.shape, train_labels.shape)
print("Val  :", val_spatial.shape, val_frequency.shape, val_labels.shape)
print("Test :", test_spatial.shape, test_frequency.shape, test_labels.shape)

# Feature-level datasets
train_feature_dataset = torch.utils.data.TensorDataset(
    train_spatial, train_frequency, train_labels
)

val_feature_dataset = torch.utils.data.TensorDataset(
    val_spatial, val_frequency, val_labels
)

test_feature_dataset = torch.utils.data.TensorDataset(
    test_spatial, test_frequency, test_labels
)

FEATURE_BATCH_SIZE = 64

train_feature_loader = DataLoader(
    train_feature_dataset,
    batch_size=FEATURE_BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_feature_loader = DataLoader(
    val_feature_dataset,
    batch_size=FEATURE_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_feature_loader = DataLoader(
    test_feature_dataset,
    batch_size=FEATURE_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("\nFeature loaders ready.")
print("Train batches:", len(train_feature_loader))
print("Val batches  :", len(val_feature_loader))
print("Test batches :", len(test_feature_loader))

Train: torch.Size([1770, 512]) torch.Size([1770, 512]) torch.Size([1770])
Val  : torch.Size([392, 512]) torch.Size([392, 512]) torch.Size([392])
Test : torch.Size([418, 512]) torch.Size([418, 512]) torch.Size([418])

Feature loaders ready.
Train batches: 28
Val batches  : 7
Test batches : 7


In [26]:


print("Initial gate:", sample_alpha[:5].squeeze().detach().cpu().numpy())
print("Gate min:", sample_alpha.min().item())
print("Gate max:", sample_alpha.max().item())

Initial gate: [0.5 0.5 0.5 0.5 0.5]
Gate min: 0.5
Gate max: 0.5


In [28]:

NUM_EPOCHS = 15

best_val_auc = -1.0
best_epoch = -1
best_state = None

train_history = []
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):


    # Training

    model.train()

    train_losses = []
    train_true = []
    train_probs = []

    for spatial, frequency, labels in train_feature_loader:

        spatial = spatial.to(device)
        frequency = frequency.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(spatial, frequency)

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

        probs = torch.softmax(logits, dim=1)[:, 1]

        train_true.extend(labels.detach().cpu().numpy())
        train_probs.extend(probs.detach().cpu().numpy())

    train_true = np.array(train_true)
    train_probs = np.array(train_probs)

    train_auc = roc_auc_score(train_true, train_probs)
    train_loss = np.mean(train_losses)


    # Validation

    model.eval()

    val_losses = []
    val_true = []
    val_probs = []

    with torch.no_grad():

        for spatial, frequency, labels in val_feature_loader:

            spatial = spatial.to(device)
            frequency = frequency.to(device)
            labels = labels.to(device)

            logits = model(spatial, frequency)

            loss = criterion(logits, labels)

            val_losses.append(loss.item())

            probs = torch.softmax(logits, dim=1)[:, 1]

            val_true.extend(labels.cpu().numpy())
            val_probs.extend(probs.cpu().numpy())

    val_true = np.array(val_true)
    val_probs = np.array(val_probs)

    val_auc = roc_auc_score(val_true, val_probs)
    val_loss = np.mean(val_losses)

    train_history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_auc": float(train_auc),
        "val_loss": float(val_loss),
        "val_auc": float(val_auc)
    })

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train AUC: {train_auc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )

    # Save best by validation AUC
    if val_auc > best_val_auc:

        best_val_auc = val_auc
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())

        print(f"  -> New best validation AUC: {best_val_auc:.4f}")

elapsed = time.time() - start_time

# Restore best model
model.load_state_dict(best_state)

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation AUC:", round(best_val_auc, 4))
print("Training time:", round(elapsed / 60, 2), "minutes")

Epoch 01/15 | Train Loss: 0.1226 | Train AUC: 0.9932 | Val Loss: 1.9311 | Val AUC: 0.8605
  -> New best validation AUC: 0.8605
Epoch 02/15 | Train Loss: 0.0335 | Train AUC: 0.9987 | Val Loss: 1.7476 | Val AUC: 0.8627
  -> New best validation AUC: 0.8627
Epoch 03/15 | Train Loss: 0.0286 | Train AUC: 0.9990 | Val Loss: 1.8033 | Val AUC: 0.8608
Epoch 04/15 | Train Loss: 0.0307 | Train AUC: 0.9988 | Val Loss: 1.4714 | Val AUC: 0.8617
Epoch 05/15 | Train Loss: 0.0245 | Train AUC: 0.9994 | Val Loss: 2.2377 | Val AUC: 0.8586
Epoch 06/15 | Train Loss: 0.0295 | Train AUC: 0.9989 | Val Loss: 1.9126 | Val AUC: 0.8605
Epoch 07/15 | Train Loss: 0.0234 | Train AUC: 0.9991 | Val Loss: 2.3231 | Val AUC: 0.8532
Epoch 08/15 | Train Loss: 0.0245 | Train AUC: 0.9992 | Val Loss: 2.4927 | Val AUC: 0.8528
Epoch 09/15 | Train Loss: 0.0245 | Train AUC: 0.9992 | Val Loss: 2.3300 | Val AUC: 0.8533
Epoch 10/15 | Train Loss: 0.0221 | Train AUC: 0.9996 | Val Loss: 2.3166 | Val AUC: 0.8541
Epoch 11/15 | Train Loss: 

In [29]:


# Restore best validation-AUC model
model.load_state_dict(best_state)
model.eval()

test_true = []
test_probs = []
test_preds = []
test_gates = []

with torch.no_grad():

    for spatial, frequency, labels in test_feature_loader:

        spatial = spatial.to(device)
        frequency = frequency.to(device)
        labels = labels.to(device)

        logits, alpha = model(
            spatial,
            frequency,
            return_gate=True
        )

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        test_true.extend(labels.cpu().numpy())
        test_probs.extend(probs.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())
        test_gates.extend(alpha.squeeze(1).cpu().numpy())

# Convert to arrays
test_true = np.array(test_true)
test_probs = np.array(test_probs)
test_preds = np.array(test_preds)
test_gates = np.array(test_gates)



test_accuracy = accuracy_score(test_true, test_preds)

test_precision = precision_score(
    test_true,
    test_preds,
    zero_division=0
)

test_recall = recall_score(
    test_true,
    test_preds,
    zero_division=0
)

test_f1 = f1_score(
    test_true,
    test_preds,
    zero_division=0
)

test_auc = roc_auc_score(
    test_true,
    test_probs
)

cm = confusion_matrix(
    test_true,
    test_preds
)

# ------------------------------------------------------------
# Gate statistics
# alpha = spatial contribution
# 1-alpha = frequency contribution
# ------------------------------------------------------------

overall_gate_mean = float(np.mean(test_gates))
overall_gate_std = float(np.std(test_gates))
overall_gate_min = float(np.min(test_gates))
overall_gate_max = float(np.max(test_gates))

real_gate = test_gates[test_true == 0]
fake_gate = test_gates[test_true == 1]

real_gate_mean = float(np.mean(real_gate))
fake_gate_mean = float(np.mean(fake_gate))

spatial_dominant_pct = float(
    np.mean(test_gates > 0.5) * 100
)

frequency_dominant_pct = float(
    np.mean(test_gates < 0.5) * 100
)


print("=" * 60)
print("NOTEBOOK 07 — ADAPTIVE GATED FUSION")
print("=" * 60)

print(f"Best Validation AUC : {best_val_auc:.4f}")
print(f"Best Epoch           : {best_epoch}")

print("\nTest Metrics")
print("-" * 40)
print(f"Accuracy             : {test_accuracy:.4f}")
print(f"Precision            : {test_precision:.4f}")
print(f"Recall               : {test_recall:.4f}")
print(f"F1 Score             : {test_f1:.4f}")
print(f"ROC-AUC              : {test_auc:.4f}")

print("\nConfusion Matrix")
print(cm)

print("\nGate Statistics")
print("-" * 40)
print(f"Overall mean alpha   : {overall_gate_mean:.4f}")
print(f"Overall std alpha    : {overall_gate_std:.4f}")
print(f"Minimum alpha        : {overall_gate_min:.4f}")
print(f"Maximum alpha        : {overall_gate_max:.4f}")
print(f"REAL mean alpha      : {real_gate_mean:.4f}")
print(f"FAKE mean alpha      : {fake_gate_mean:.4f}")
print(f"Spatial dominant     : {spatial_dominant_pct:.2f}%")
print(f"Frequency dominant   : {frequency_dominant_pct:.2f}%")


MODEL_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
CM_DIR = os.path.join(RESULTS_DIR, "confusion_matrices")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CM_DIR, exist_ok=True)

GATED_CHECKPOINT = os.path.join(
    MODEL_DIR,
    "gated_fusion_resnet18_auc_best.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "best_epoch": best_epoch,
        "best_val_auc": best_val_auc,
        "architecture": "FeatureGatedFusion",
        "feature_dim": 512,
        "source_backbone": "ResNet18",
        "fusion": "adaptive_gated",
    },
    GATED_CHECKPOINT
)


CM_FILE = os.path.join(
    CM_DIR,
    "gated_fusion_confusion_matrix.npy"
)

np.save(CM_FILE, cm)

RESULTS_FILE = os.path.join(
    RESULTS_DIR,
    "gated_fusion_results.json"
)

results = {
    "model": "Adaptive Gated Spatial-Frequency Fusion",
    "backbone": "ResNet18",
    "feature_dimension": 512,
    "fusion_type": "adaptive_gated",
    "best_epoch": int(best_epoch),
    "best_validation_auc": float(best_val_auc),

    "test_metrics": {
        "accuracy": float(test_accuracy),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
        "roc_auc": float(test_auc)
    },

    "confusion_matrix": cm.tolist(),

    "gate_statistics": {
        "overall_mean": overall_gate_mean,
        "overall_std": overall_gate_std,
        "minimum": overall_gate_min,
        "maximum": overall_gate_max,
        "real_mean": real_gate_mean,
        "fake_mean": fake_gate_mean,
        "spatial_dominant_percent": spatial_dominant_pct,
        "frequency_dominant_percent": frequency_dominant_pct
    },

    "training_history": train_history
}

with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 60)
print("ARCHIVED SUCCESSFULLY")
print("=" * 60)
print("Checkpoint :", GATED_CHECKPOINT)
print("Results    :", RESULTS_FILE)
print("Confusion  :", CM_FILE)

NOTEBOOK 07 — ADAPTIVE GATED FUSION
Best Validation AUC : 0.8627
Best Epoch           : 2

Test Metrics
----------------------------------------
Accuracy             : 0.8565
Precision            : 0.9106
Recall               : 0.9209
F1 Score             : 0.9157
ROC-AUC              : 0.8038

Confusion Matrix
[[ 32  32]
 [ 28 326]]

Gate Statistics
----------------------------------------
Overall mean alpha   : 0.9999
Overall std alpha    : 0.0002
Minimum alpha        : 0.9979
Maximum alpha        : 1.0000
REAL mean alpha      : 0.9999
FAKE mean alpha      : 0.9999
Spatial dominant     : 100.00%
Frequency dominant   : 0.00%

ARCHIVED SUCCESSFULLY
Checkpoint : /content/drive/MyDrive/Multimedia_Authentication/models/gated_fusion_resnet18_auc_best.pth
Results    : /content/drive/MyDrive/Multimedia_Authentication/results/gated_fusion_results.json
Confusion  : /content/drive/MyDrive/Multimedia_Authentication/results/confusion_matrices/gated_fusion_confusion_matrix.npy
